# 03 - Build the FAISS knowledge base

The retrieval module uses 768-dimensional mean-pooled `roberta-base`
embeddings of 768,683 labelled YelpNYC and YelpZIP reviews. Embeddings are
L2-normalised and indexed with an exact inner-product FAISS index, which is
equivalent to cosine similarity.

> The paper describes `IndexFlatL2`; the released implementation uses
> `IndexFlatIP` over normalised vectors. See `src/spam_detection/retrieval.py`.

The full run requires a GPU and roughly 3 GB of disk space. The resulting
parquet and index files are excluded from Git.

## Setup

In [ ]:
import numpy as np
import pandas as pd

from spam_detection.config import (
    FAISS_DIR,
    VECTOR_DB_INDEX,
    VECTOR_DB_PARQUET,
    YELPNYC_REVIEWS_PATH,
    YELPZIP_PATH,
)
from spam_detection.embeddings import get_roberta_embeddings, load_roberta_encoder
from spam_detection.preprocessing import build_vector_db_frame, prepare_yelp_zip
from spam_detection.retrieval import (
    build_faiss_index,
    load_labeled_faiss_db,
    summary_from_faiss,
)
from spam_detection.utils import set_seed

set_seed(42)
FAISS_DIR.mkdir(parents=True, exist_ok=True)

## Load and normalise YelpNYC / YelpZIP

In [ ]:
nyc = pd.read_parquet(YELPNYC_REVIEWS_PATH)
if YELPZIP_PATH.suffix == ".parquet":
    yelpzip = pd.read_parquet(YELPZIP_PATH)
else:
    yelpzip = prepare_yelp_zip(pd.read_csv(YELPZIP_PATH))

print("YelpNYC:", nyc.shape, list(nyc.columns))
print("YelpZIP:", yelpzip.shape, list(yelpzip.columns))

In [ ]:
knowledge_base = build_vector_db_frame(nyc, yelpzip)
knowledge_base.to_parquet(VECTOR_DB_PARQUET, index=False)

print("Combined reviews:", len(knowledge_base))
print(knowledge_base["label"].value_counts())
print(knowledge_base["label"].value_counts(normalize=True).round(4))
display(knowledge_base.head(3))

## Extract RoBERTa embeddings

This is the expensive step. Run it on a GPU if possible. The embeddings can be
saved to `data/faiss/Yelp_NYC_ZIP_roberta_embeddings.npy` so the index can be
rebuilt without re-encoding the corpus.

In [ ]:
EMBEDDINGS_PATH = FAISS_DIR / "Yelp_NYC_ZIP_roberta_embeddings.npy"

if EMBEDDINGS_PATH.exists():
    embeddings = np.load(EMBEDDINGS_PATH)
else:
    encoder, tokenizer = load_roberta_encoder(device="cuda")
    embeddings = get_roberta_embeddings(
        knowledge_base["review_text"].tolist(),
        model=encoder,
        tokenizer=tokenizer,
        batch_size=32,
        max_length=512,
        show_progress=True,
    )
    np.save(EMBEDDINGS_PATH, embeddings)

print(embeddings.shape, embeddings.dtype)

## Build the FAISS index

In [ ]:
index = build_faiss_index(embeddings, normalize=True, index_path=VECTOR_DB_INDEX)
print("Vectors indexed:", index.ntotal)
print("Dimensions:", index.d)

## Verify retrieval

The scores returned by the inner-product index are cosine similarities:
higher values mean more similar reviews.

In [ ]:
faiss_db = load_labeled_faiss_db(
    VECTOR_DB_PARQUET,
    VECTOR_DB_INDEX,
    columns=("review_text", "label"),
)

sample_index = 0
summary = summary_from_faiss(faiss_db, embeddings[sample_index], k=20)
print("Query label:", knowledge_base.iloc[sample_index]["label"])
print("Top-1 similarity:", summary["top1_score"])
print("Fake ratio in top-20:", summary["fake_ratio"])
for example in summary["top3_examples"]:
    print("-", example[:180])